# Latent Monitor Audit: SIM-CoT CODI-1B on `asdiv` (ledger §123)

## tl;dr

One command, one card. **No training.** The driver runs the twin audit on both profiles, the
paired bootstrap, the masking audit with carrying slots derived from the long twins, and the
monitor-faithfulness analysis, then writes `card.json` / `card.md` with the environment and
predictions fingerprints so cards can be compared across models, pools and GPUs.

Pool `asdiv`: EleutherAI/asdiv (Miao, Liang & Su 2020; CC BY-NC 4.0); the formula's expression tree evaluated in order as the chain. 
No dataset to attach; the gate is the full GSM8K reproduction.
When finished, publish the output as a dataset (Output tab -> New Dataset) so the card and predictions are kept.

## Setup

In [ ]:
REPO_URL = "https://github.com/0x0shephard/latent-reasoning.git"
RUN_COMMIT = "ce649b17d487fd16877c6d7c1ff536087f05dab8"
REPO_DIR = "/kaggle/working/latent-reasoning"
CONFIG = "configs/official_simcot_codi_llama1b.yaml"
MODEL = "simcot"
POOL = "asdiv"
OUTPUT_ROOT = "/kaggle/working/lma_simcot_asdiv"
RUN_SMOKE = True
BATCH_SIZE = "32"
PRECISIONS = ["float16", "float32"]
import torch
if MODEL == "codi8b":
    assert torch.cuda.device_count() >= 2, "the 8B audit needs both T4s (accelerator GPU T4 x2)"

import glob, json, os, pathlib, subprocess, sys, time
os.environ.setdefault("HF_HUB_DISABLE_XET", "1")
os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "300")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
assert len(RUN_COMMIT) == 40 and RUN_COMMIT != "0" * 40
if not pathlib.Path(REPO_DIR).exists():
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
subprocess.run(["git", "-C", REPO_DIR, "fetch", "--all", "--tags"], check=True)
subprocess.run(["git", "-C", REPO_DIR, "checkout", RUN_COMMIT], check=True)
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print("code commit", subprocess.run(["git", "rev-parse", "HEAD"], check=True, capture_output=True, text=True).stdout.strip())
print("gpu", subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip())

### Environment and focused tests

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "tpot"], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "transformers==4.52.4", "datasets==3.6.0", "peft==0.15.2", "accelerate==1.7.0",
                "huggingface-hub>=0.34,<1.0", "safetensors==0.5.3", "pytest>=8,<10"], check=True)
probe = subprocess.run([sys.executable, "-c",
    "from peft.import_utils import is_torchao_available\ntry:\n print(is_torchao_available())\nexcept ImportError:\n print('incompatible')"],
    capture_output=True, text=True)
if "incompatible" in probe.stdout:
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=True)
subprocess.run([sys.executable, "-m", "pytest", "-q", *["tests/test_official_codi_family.py", "tests/test_cache_carrier.py", "tests/test_directed_counterfactual_runner.py", "tests/test_masking_audit_runner.py", "tests/test_monitor_faithfulness.py", "tests/test_directed_counterfactual_bootstrap.py", "tests/test_chain_pools.py", "tests/test_monitor_audit.py", "tests/test_latent_monitor_audit_runner.py", "tests/test_masking_bootstrap.py"]], check=True)

def run_with_retry(command, attempts=3, wait=60):
    for attempt in range(1, attempts + 1):
        result = subprocess.run(command)
        if result.returncode == 0:
            return
        if attempt == attempts:
            raise subprocess.CalledProcessError(result.returncode, command)
        print(f"runner exited with {result.returncode}; retrying in {wait}s ({attempt + 1}/{attempts})")
        time.sleep(wait)

## Step 1: gate

In [ ]:
def summary_under(root):
    found = sorted(glob.glob(f"{root}/eval/revision_*/full_*/summary.json"), key=os.path.getmtime)
    return found[-1] if found else None

import yaml
cfg = yaml.safe_load(open(CONFIG)); PAPER = cfg["accuracy_gate"]["published_accuracy"]["gsm8k"]
PRECISION = None
for precision in PRECISIONS:
    run_with_retry([sys.executable, "-u", "-m", "src.eval.official_codi", "--config", CONFIG, "--device", "cuda",
                    "--set", f"model.dtype={precision}", "--set", f"eval.batch_size={BATCH_SIZE}"])
    REPRODUCTION_SUMMARY = summary_under(cfg["output_dir"])
    gate = json.loads(open(REPRODUCTION_SUMMARY).read())
    acc = gate["datasets"]["gsm8k"]
    print(f"{precision}: gsm8k {acc:.4f} (paper {PAPER}, delta {acc - PAPER:+.4f}); gate {gate['accuracy_gate']['status']}")
    if gate["accuracy_gate"]["status"] == "passed":
        PRECISION = precision
        break
assert PRECISION, "gate failed in both precisions"

## Step 2: the audit (smoke first, then full)

In [ ]:
def audit(out, smoke):
    cmd = [sys.executable, "-u", "scripts/run_latent_monitor_audit.py", "--config", CONFIG, "--pool", POOL,
           "--reproduction-summary", REPRODUCTION_SUMMARY, "--output-dir", out, "--batch-size", BATCH_SIZE,
           "--precision", PRECISION, "--device", "cuda", "--model-name", MODEL]
    if smoke:
        cmd.append("--smoke")
    run_with_retry(cmd)
    return json.loads(open(pathlib.Path(out) / "card.json").read())

if RUN_SMOKE:
    audit(OUTPUT_ROOT + "_smoke", True)
card = audit(OUTPUT_ROOT, False)

## The card

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open(pathlib.Path(OUTPUT_ROOT) / "card.md").read()))
print(json.dumps(card["verdict"], indent=2)); print(json.dumps(card["checks"], indent=2))
print("environment", card["environment"]); print("fingerprints", card["fingerprints"]); print("stage seconds", card["stage_seconds"])
print("Reminder: publish this notebook's output as a dataset (Output tab -> New Dataset).")

## Keep the output under Kaggle's 500-file cap

In [ ]:
import shutil
shutil.rmtree(REPO_DIR, ignore_errors=True)   # the clone (with .git) is not an output; without this the run's own files can be dropped from the saved output
print("removed", REPO_DIR, "; remaining output files:", sum(len(f) for _, _, f in os.walk("/kaggle/working")))